# Suite VAL — Validators

`Validators.Validate(store, constraints)` evaluates each object's constraints, those of its own schema, and reports
what does not hold, what is unknown and what raised, labelled as mbse-schemas labels objects.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Patterns import Constraints as C, Validators as V
from support import Contact, Phone, book, raises

this = E.variable("this")
constraints = C.Set([
    C.Constraint("Contact", "adult", this.age.ge(18)),
    C.Constraint("Contact", "phoned", E.operation("count", E.operation("entries", this, "phones")).ge(1)),
    C.Constraint("Phone", "numbered", this.has("number")),
])
store = book()
ann = store.Contact().name("ann").age(30).phones(lambda e: e.phone(lambda p: p.number("555"))).create()
bob = store.Contact().name("bob").phones(lambda e: e.phone(lambda p: p)).create()

## VAL-01 · Each object is checked against its schema's constraints

In [ ]:
validate = V.Validate(store, constraints)
assert validate(Contact, ann) == []
assert validate(Contact, bob) == ["Contact#0: 'adult' is unknown"]  # bob's age is absent
young = store.Contact().name("cy").age(9).create()
assert validate(Contact, young) == ["Contact#0: 'adult' does not hold", "Contact#0: 'phoned' does not hold"]
assert validate(Phone, store.Phone().create()) == ["Phone#0: 'numbered' does not hold"]
assert validate(Contact, store.Phone().create()) == ["the value is a 'Phone', not an instance of the given schema"]

## VAL-02 · Reachable checks every object reachable from the root

In [ ]:
assert V.Validate(store, constraints).Reachable(Contact, bob) == ["Contact#0: 'adult' is unknown",
                                                                   "Phone#1: 'numbered' does not hold"]
assert V.Validate(store, constraints).Reachable(Contact, ann) == []

## VAL-03 · An unknown result is reported, ignored or a violation

In [ ]:
assert V.Validate(store, constraints, unknown="ignore")(Contact, bob) == []
assert V.Validate(store, constraints, unknown="violation")(Contact, bob) == ["Contact#0: 'adult' does not hold"]
assert V.Validate(store, constraints, unknown="report")(Contact, bob) == ["Contact#0: 'adult' is unknown"]
with raises(ValueError, match="unknown must be 'report', 'ignore' or 'violation', got 'maybe'"):
    V.Validate(store, constraints, unknown="maybe")
assert V.UNKNOWN == ("report", "ignore", "violation")

## VAL-04 · A rule that raises is reported in place; a rule that cannot be right is refused

In [ ]:
raising = C.Set([C.Constraint("Contact", "mixed", E.operation("add", this.age, 1.5).ge(2.0)),
                 C.Constraint("Contact", "number", this.age)])
assert V.Validate(store, raising)(Contact, ann) == [
    "Contact#0: 'mixed' raised TypeError: add expects numbers of one domain, got int and float",
    "Contact#0: 'number' raised TypeError: a predicate must be a bool, got int"]
with raises(ValueError, match="constraint 'x' of 'Contact': variable 'n' is not bound"):
    V.Validate(store, [C.Constraint("Contact", "x", E.variable("n"))])